In [ ]:
import os
import sys
import pandas  as pd
import numpy as np
import yfinance as yf
import scipy.stats as stats
import numpy_financial as npf
import matplotlib.pyplot as plt
import matplotlib
import scipy

#查看当前目录
current_dir=os.getcwd()
#查看版本
print(f'当前python版本为：{sys.version}')
print(f'当前pandas版本为：{pd.__version__}')
#查看库中可用函数
print([i for i in dir(pd) if not i.startswith('_')])


In [ ]:
##查看股票收盘价
tickers=['AAPL','MSFT','GOOGL']
startdate='2022-01-01'
data=yf.download(tickers,startdate)
#print(data.columns)
close_data=data['Close']
#数据清洗：缺失值填充
close_data=close_data.dropna()#直接删除空值
close_data_clean=close_data.ffill() #用前一天的数值填充
#计算对数收益率
log_return=np.log(close_data_clean/close_data_clean.shift(1)).dropna()
#计算季度收益率,是在对数收益率的基础上进行分组计算
return2=log_return.copy()
#print(return2.index)
return2['year']=return2.index.year
return2['quarter']=return2.index.quarter
quarterly_rate=return2.groupby(['year','quarter']).mean()
#选择两只股票进行双边t检验，t_stat反映 AAPL 与 MSFT 收益率均值差异的大小，
#p_value用于判断该差异是否具有统计显著性，小于0.05差异显著
aapl_returns = log_return['AAPL'].dropna()
msft_returns = log_return['MSFT'].dropna()
t_stat,p_value=stats.ttest_ind(aapl_returns, msft_returns)

In [ ]:
#计算每只股票的夏普比率 Sharpe = (Mean Return - Risk Free Rate) / Std Dev
risk_free_rate = 0.01 / 252
Sharpe_ratio={}
for ticker in tickers:
    mean_return=log_return[ticker].mean()
    std_return=log_return[ticker].std()
    sharpe = (mean_return - risk_free_rate) / std_return
    Sharpe_ratio[ticker] = sharpe*np.sqrt(252) #年化夏普比率
print("夏普比率：", Sharpe_ratio)


In [ ]:
#计算累计收益率并可视化
cumulative_returns = (1 + log_return).cumprod()
for ticker in tickers:
    plt.plot(cumulative_returns.index, cumulative_returns[ticker], 
             label=ticker, linewidth=2)
plt.title('Cumulative Returns Comparison (2022-Present)', fontsize=16)
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Return', fontsize=12)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
#整体展示
summary = pd.DataFrame({
    'Ticker': tickers,
    'Sharpe': [Sharpe_ratio[t] for t in tickers],
    'Mean_Ret': [log_return[t].mean() for t in tickers],
    'Std': [log_return[t].std() for t in tickers]
})
